# 04.3 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/03-missing-and-outliers/exercises.ipynb)

Run the setup cell first.

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail()                                   # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

def customer_table(cutoff, months=3):
    """One row per customer: features from before `cutoff`, churned = no order in the next `months`."""
    cutoff = pd.Timestamp(cutoff)
    past = sales[sales["InvoiceDate"] < cutoff]
    future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < cutoff + pd.DateOffset(months=months))]
    c = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                        products=("StockCode", "nunique"), country=("Country", "first"),
                                        first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
    c["tenure_days"] = (cutoff - c["first_order"]).dt.days
    c["days_since_last"] = (cutoff - c["last_order"]).dt.days
    came_back = future.groupby("Customer ID")["Revenue"].sum().reindex(c.index).fillna(0) > 0
    c["churned"] = (~came_back).astype(int)
    return c.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
cust = customer_table("2011-09-01")
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

past = sales[sales["InvoiceDate"] < "2011-09-01"]
span = past.groupby("Customer ID")["InvoiceDate"].agg(["min", "max"])
days_active = (span["max"] - span["min"]).dt.days.reindex(cust.index)
# average days between orders; undefined (missing) for customers with only one order
cust["avg_gap_days"] = (days_active / (cust["orders"] - 1)).where(cust["orders"] > 1)
print(f"avg_gap_days missing for {cust['avg_gap_days'].isna().mean():.1%} of customers")

gap = pd.Series([12, 20, np.nan, 30, np.nan, 45, 90, np.nan], name="avg_gap_days")
country = pd.Series(["UK", "UK", "DE", np.nan, "UK", "FR", np.nan, "UK"], name="country")

spend8 = pd.Series([120, 150, 180, 200, 240, 260, 300, 4800], name="spend")

3,317 customers, churn rate 43.6%
avg_gap_days missing for 41.5% of customers


## Exercise 1: by hand

For `gap` (12, 20, ?, 30, ?, 45, 90, ?): suppose only the first five rows are the training set. Which mean would `SimpleImputer(strategy="mean")` learn? Fill the three gaps with it. Check in code.

In [2]:
# your code here

## Exercise 2: what does missing mean?

In the raw transactions, `Description` is missing for some rows. Look at those rows: what are their `Price`, `Quantity` and `Customer ID`? Is this MCAR, MAR or MNAR, and what would you do with them?

In [3]:
# your code here

## Exercise 3: the z-score after removing the wholesaler

Remove 4,800 from `spend8` and recompute the mean, standard deviation and the z-score that 4,800 *would* have had. Is it now above 3?

In [4]:
# your code here

## Exercise 4: trees and imputation

Fill `avg_gap_days` with three constants (−1, 0, 999) and train a `DecisionTreeClassifier(max_depth=4)` on `days_since_last` and `avg_gap_days`. Does the choice of constant matter for a tree? Why or why not?

In [5]:
# your code here